##  Streamlit: ML web app

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import joblib

## Load dataset and inspect

    Used dataset from:  Exploratory Data Analysis project Airbnb in New York

In [2]:
# load out dataset
df = pd.read_csv('https://raw.githubusercontent.com/4GeeksAcademy/data-preprocessing-project-tutorial/main/AB_NYC_2019.csv')

df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [3]:
df.shape

(48895, 16)

In [4]:
print(df.isna().sum())

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64


## Let's clean up our dataset and prepare it for my App

In [5]:
# Standardize empty/blank values
df = df.replace(r'^\s*$', np.nan, regex=True)

In [6]:
# determine what we are keeping as our X (predictors) and y (target)
target = 'price'
numeric_feat = ['minimum_nights', 'number_of_reviews', 'calculated_host_listings_count']
categorical_feat = ['room_type', 'neighbourhood_group']

keep_cols = ([target] + numeric_feat + categorical_feat)

df=df[keep_cols]

df.head()

,price,minimum_nights,number_of_reviews,calculated_host_listings_count,room_type,neighbourhood_group
0,149,1,9,6,Private room,Brooklyn
1,225,1,45,2,Entire home/apt,Manhattan
2,150,3,0,1,Private room,Manhattan
3,89,1,270,1,Entire home/apt,Brooklyn
4,80,10,9,1,Entire home/apt,Manhattan


In [7]:
# Confirm missing values
print(df.isna().sum())

price                             0
minimum_nights                    0
number_of_reviews                 0
calculated_host_listings_count    0
room_type                         0
neighbourhood_group               0
dtype: int64


In [8]:
# Drop rows with missing values in selected columns
df = df.dropna()

In [9]:
# Safety check
assert df.isna().sum().sum() == 0, 'NaNs still present after cleaning!'


In [ ]:
# converts categorical text columns into numeric col so ML mode can use them
df_encoded = pd.get_dummies(df, columns=categorical_feat, drop_first=True)

## Train our model

In [11]:
# define X and y
X = df_encoded.drop(columns=[target])
y = df_encoded[target]

In [12]:
# features will be reused hwen making predictions in Streamlit
features = X.columns.tolist()

In [13]:
# create out model
linreg_model = LinearRegression()

# apply fit to our model so it can teach the model to make predict prices
linreg_model.fit(X,y)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [14]:
# use the trained model to generate predicted prices
y_preds = linreg_model.predict(X)

In [15]:
r2 = r2_score(y, y_preds)
mae = mean_absolute_error(y, y_preds)

print('\nMODEL EVALUATION SUMMARY')
print('-' * 40)
print(f'R2 Score (variance explained): {r2:.3f}')
print(f'Mean Absolute Error (USD):     ${mae:,.2f}')
print('-' * 40)


MODEL EVALUATION SUMMARY
----------------------------------------
R2 Score (variance explained): 0.083
Mean Absolute Error (USD):     $73.61
----------------------------------------


## Save our model

In [16]:
# Define the filename
filename = 'finalized_model.joblib'

In [ ]:
# dictionary that contains the trained model and everything needed to use it correctly later
joblib.dump({
        'model': linreg_model,
        'features': features,
        'metrics':{
            'r2': r2,
            'mae': mae
        }
    },
    filename
)

['finalized_model.joblib']

In [18]:
features_list = ['calculated_host_listings_count', 'minimum_nights', 'number_of_reviews']

In [19]:
# user wants host to have 2 houses
# and minimum nights 1
# and 45 reviews

## When opening this project you must activate this virtual environment by running the following in the terminal:

    - source .venv/bin/activate
    - streamlit run src/Streamlit_app.py

## To stop the app and gain control of terminal:

    - Hit Control C

